# ETI Research Core v0.2 — budget-aware CPU screening
Run only when intentionally starting a new experiment. This notebook uses public market data and never places orders.

## User-set Kaggle spending ceilings
- Daily ceiling: USD 10
- Monthly ceiling: USD 100
- These are hard limits, not targets. Check Kaggle's own usage/billing display before running; this notebook cannot read or enforce dollar billing.
- CPU only. One symbol. No hyperparameter sweep. Estimated model-fit calls are capped before training.

## Actual model export
- The run automatically tries Coinbase Exchange then Kraken for public BTC spot candles; USDT symbols are explicitly mapped to BTC/USD and the actual source is recorded. The run reserves the final 20% of candles from nested research evaluation, then fits one deployable BTC spot candidate using chronological 60/20/20 classifier-fit, temperature-calibration and untouched probability-validation blocks with label-horizon purge gaps.
- The candidate is saved as a content-addressed `.joblib` artifact under `artifacts/models/`, with SHA-256 and feature-schema metadata in the JSON report.
- This is a research candidate, not approval for live trading.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/eti-research-core')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/hamza00alik-source/eti-research-core.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
%cd /kaggle/working/eti-research-core
%pip install -r requirements.txt


In [ ]:
import os
from pathlib import Path
import sys

# First-stage screening: intentionally small and CPU-only.
EXCHANGE = 'auto'
SYMBOL = 'BTC/USDT'
TIMEFRAME = '1h'
CANDLE_LIMIT = 2000
HORIZON = 6
MIN_TRAIN = 400
VALIDATION_SIZE = 100
TEST_SIZE = 100
STEP_SIZE = 200
MAX_MODEL_FITS = 32
OUTPUT = 'artifacts/eti_budget_screen.json'
MODEL_DIR = 'artifacts/models'
os.environ['ETI_ARTIFACT_DIR'] = MODEL_DIR

command = [
    sys.executable, '-m', 'eti.pipeline',
    '--exchange', EXCHANGE, '--symbol', SYMBOL,
    '--timeframe', TIMEFRAME, '--limit', str(CANDLE_LIMIT),
    '--horizon', str(HORIZON), '--min-train', str(MIN_TRAIN),
    '--validation-size', str(VALIDATION_SIZE),
    '--test-size', str(TEST_SIZE), '--step-size', str(STEP_SIZE),
    '--max-model-fits', str(MAX_MODEL_FITS),
    '--evaluation', 'nested', '--output', OUTPUT,
    '--save-model', '--model-id', 'eti-btc-1h', '--model-version', '0.1.0',
]
print('Starting one BTC spot CPU screening run with Coinbase Exchange -> Kraken fallback; dollar spend must be checked in Kaggle UI.')
subprocess.run(command, check=True)
print(Path(OUTPUT).read_text())
print('Model artifact files:')
for path in sorted(Path(MODEL_DIR).rglob('*.joblib')):
    print(path, path.stat().st_size, 'bytes')

# Fail the run unless the current canonical research output is complete and auditable.
import hashlib, json
report = json.loads(Path(OUTPUT).read_text())
assert report.get('training_performed') is True, 'Nested model fitting was not confirmed'
assert report.get('execution_enabled') is False, 'Research artifact must explicitly disable execution'
assert report.get('rows', 0) >= 662, f"Insufficient usable research rows: {report.get('rows')}"
assert report.get('fold_count', 0) >= 1, 'No outer-test folds were evaluated'
folds = report.get('folds')
assert isinstance(folds, list) and len(folds) == report['fold_count'], 'Fold count does not match fold evidence'
budget = report.get('compute_budget', {})
assert budget.get('total_estimated_model_fit_calls', 10**9) <= MAX_MODEL_FITS, 'Model-fit cap exceeded'
assert budget.get('max_model_fit_calls') == MAX_MODEL_FITS, 'Unexpected model-fit cap'
source_info = report.get('market_data', {})
assert source_info.get('provider') in {'coinbaseexchange', 'kraken'}, f"Unexpected provider: {source_info.get('provider')}"
assert source_info.get('requested_symbol') == SYMBOL and source_info.get('venue_symbol') == 'BTC/USD', 'Unexpected market mapping'
assert source_info.get('returned_candles', 0) >= 662, 'Provider returned too few completed candles'
exported = report.get('exported_model', {})
digest = exported.get('artifact_sha256')
assert isinstance(digest, str) and len(digest) == 64, 'Missing model SHA-256'
matches = list(Path(MODEL_DIR).rglob(digest + '.joblib'))
assert len(matches) == 1, f'Expected one content-addressed model file, got {len(matches)}'
assert hashlib.sha256(matches[0].read_bytes()).hexdigest() == digest, 'Model artifact checksum mismatch'
assert exported.get('calibration_status') in {'calibrated', 'uncalibrated'}, 'Missing explicit calibration status'
independent = (exported.get('input_schema') or {}).get('independent_probability_validation', {})
assert isinstance(independent.get('passed'), bool), 'Missing independent probability-validation result'
scenarios = (report.get('cost_sensitivity') or {}).get('scenarios', [])
assert any(item.get('cost_multiplier') == 2.0 and item.get('fold_count') == report['fold_count'] for item in scenarios), 'Missing 2x-cost outer-test evidence'
print('ETI TRAINING READINESS: PASS')
print(json.dumps({
    'provider': source_info['provider'],
    'venue_symbol': source_info['venue_symbol'],
    'candles': source_info['returned_candles'],
    'usable_rows': report['rows'],
    'outer_test_folds': report['fold_count'],
    'model_fit_calls': budget['total_estimated_model_fit_calls'],
    'model_sha256': digest,
    'calibration_status': exported['calibration_status'],
    'independent_probability_validation_passed': independent['passed'],
    'execution_enabled': False,
}, indent=2))
